# EmprendeCopy
## POC — Fast Prompting en Acción

**Curso:** 95920 – Inteligencia artificial: Generación de Prompts  
**Alumno:** Diego Javier Pérez  
**Preentrega 2:** Fast Prompting en Acción: Desentrañando la Magia

---

### Propósito

En esta notebook continúo con **EmprendeCopy**, el proyecto que planteé en la primera preentrega.

La idea de esta POC es probar cómo cambia el resultado cuando paso de un prompt simple a una técnica de **few-shot prompting**, y ver si eso me ayuda a conseguir respuestas más consistentes para publicaciones de pequeños negocios.


## 1. Problema y propuesta

Muchos emprendimientos y pequeños negocios usan las redes sociales para mostrar promociones, productos o servicios, pero no siempre tienen tiempo o experiencia en redacción publicitaria y diseño.

La idea de **EmprendeCopy** es que el usuario cargue algunos datos básicos de su negocio y obtenga:

- un título;
- un copy promocional;
- una llamada a la acción (CTA);
- cinco hashtags;
- un prompt visual en inglés para generar una imagen.

En esta POC voy a comparar **zero-shot y few-shot** usando el mismo caso, y al final voy a mostrar una versión más optimizada de la solución.


## 2. Objetivos de la POC

Con esta prueba busco:

1. Hacer una implementación funcional usando Groq.
2. Comparar un prompt zero-shot con uno few-shot.
3. Ver si los ejemplos ayudan a mantener mejor el formato de las respuestas y medir esa mejora con una escala de 0 a 10.
4. Usar la prueba de imagen de la primera entrega para mostrar cómo se puede mejorar un prompt a partir del resultado.
5. Dejar una versión final que genere el copy y el prompt visual con una sola consulta.
6. Evitar consultas innecesarias y mantener la API key fuera del repositorio.


## 3. Instalación

Primero instalo el SDK de Groq. En Colab esta celda se puede ejecutar directamente.


In [1]:
%pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


## 4. Configuración de Groq

Para no dejar la API key escrita en el notebook, la guardé en los **Secrets de Google Colab** con el nombre `GROQ_API_KEY`.

Si se ejecuta en un entorno local, también se puede leer desde una variable de entorno con ese mismo nombre.

> De esta forma la clave no queda expuesta cuando el repositorio se sube a GitHub.


In [2]:
import os
import json
from groq import Groq

# Primero intento leer la key desde Colab; si no estoy en Colab, busco una variable de entorno.
api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    raise RuntimeError(
        "No se encontró GROQ_API_KEY. "
        "En Colab agregala en Secrets; en local definila como variable de entorno."
    )

# Uso el mismo modelo que se trabajó en el material del curso.
MODELO = "openai/gpt-oss-20b"

client = Groq(api_key=api_key)

print("Groq listo.")
print("Modelo:", MODELO)


Groq listo.
Modelo: openai/gpt-oss-20b


## 5. Función para consultar el modelo

Para no repetir el mismo código en cada prueba armé una función que hace la consulta y, además de devolver la respuesta, guarda la cantidad de tokens utilizados.

Con esos datos después puedo hacer una estimación simple del costo de cada prueba.

Uso `reasoning_effort="low"` porque en este caso la tarea es bastante directa y no necesita un nivel de razonamiento alto.


In [4]:
# Precios que usé para estimar el costo de las pruebas.
# Los consulté en la documentación de Groq el 23/08/2026.
PRECIO_INPUT_USD_M = 0.075
PRECIO_OUTPUT_USD_M = 0.30

def estimar_costo(usage):
    """Estima cuánto costó una consulta usando los tokens informados por Groq."""
    input_tokens = getattr(usage, "prompt_tokens", 0) or 0
    output_tokens = getattr(usage, "completion_tokens", 0) or 0

    costo_input = (input_tokens / 1_000_000) * PRECIO_INPUT_USD_M
    costo_output = (output_tokens / 1_000_000) * PRECIO_OUTPUT_USD_M

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "costo_estimado_usd": costo_input + costo_output,
    }


def chat(user, system="Respondé en español, de forma clara y breve.", temperature=0):
    """Hace una consulta a Groq y devuelve la respuesta junto con los datos de uso."""
    respuesta = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
        reasoning_effort="low",
        max_completion_tokens=1200,
    )

    texto = respuesta.choices[0].message.content
    metricas = estimar_costo(respuesta.usage)

    return texto, metricas


## 6. Caso de prueba

Para las comparaciones voy a usar el mismo ejemplo de la primera preentrega. Así puedo mantener una continuidad con el proyecto y comparar los cambios sobre un caso conocido.

| Variable | Valor |
|---|---|
| Negocio | Peluquería femenina |
| Servicio | Coloración |
| Promoción | 20 % de descuento durante agosto |
| Público | Mujeres de 20 a 50 años |
| Plataforma | Instagram |
| Tono | Cercano y profesional |
| Objetivo | Conseguir reservas de turnos |


In [5]:
caso = {
    "tipo_negocio": "Peluquería femenina",
    "producto_servicio": "Coloración",
    "promocion": "20 % de descuento durante agosto",
    "publico": "Mujeres de 20 a 50 años",
    "plataforma": "Instagram",
    "tono": "Cercano y profesional",
    "objetivo": "Conseguir reservas de turnos",
}

caso


{'tipo_negocio': 'Peluquería femenina',
 'producto_servicio': 'Coloración',
 'promocion': '20 % de descuento durante agosto',
 'publico': 'Mujeres de 20 a 50 años',
 'plataforma': 'Instagram',
 'tono': 'Cercano y profesional',
 'objetivo': 'Conseguir reservas de turnos'}

# Parte A — Zero-shot vs Few-shot

## 7. Primera prueba: zero-shot

Empiezo con un prompt **zero-shot**, es decir, le doy al modelo la tarea y los datos, pero no le muestro ejemplos previos de cómo quiero la respuesta.

Esta primera prueba me sirve como punto de comparación para ver qué cambia cuando agrego ejemplos.


In [6]:
prompt_zero_shot = f"""
Creá una publicación promocional para {caso['plataforma']} para una {caso['tipo_negocio']}.
Servicio: {caso['producto_servicio']}.
Promoción: {caso['promocion']}.
Público objetivo: {caso['publico']}.
Tono: {caso['tono']}.
Objetivo: {caso['objetivo']}.

Incluí un título, un copy breve, una llamada a la acción y hashtags.
"""

respuesta_zero, metricas_zero = chat(prompt_zero_shot, temperature=0)

print("=== RESPUESTA ZERO-SHOT ===")
print(respuesta_zero)
print("\n=== USO ===")
print(metricas_zero)


=== RESPUESTA ZERO-SHOT ===
**✨¡Renueva tu look este agosto!✨**  

En nuestra peluquería, la coloración profesional ahora con **20 % de descuento** solo en agosto.  
Mujeres de 20 a 50 años, es tu momento de brillar con un color que refleje tu estilo.

📅 Reserva tu turno hoy y descubre la diferencia de un color impecable y cuidado.  

👉 **Haz clic en el enlace de nuestra bio o envíanos un DM** para agendar tu cita.  

#PeluqueríaFemenina #Coloración #DescuentoAgosto #MujeresConEstilo #ReservaYa #BellezaProfesional #CuidadoCapilar #Tendencias2026 #HairColor #AgostoConEstilo

=== USO ===
{'input_tokens': 168, 'output_tokens': 172, 'total_tokens': 340, 'costo_estimado_usd': 6.42e-05}


### Qué miro en esta primera prueba

El resultado puede ser correcto, pero hay algunas cosas que el modelo decide por su cuenta:

- cuántos hashtags usar;
- cómo ordenar las secciones;
- si agrega algún texto extra;
- qué extensión darle a cada parte.

Ahora voy a repetir el mismo caso, pero usando ejemplos para marcar mejor el formato que quiero.


## 8. Segunda prueba: few-shot prompting

En la segunda prueba uso **few-shot prompting**.

La diferencia es que antes del caso nuevo incluyo dos ejemplos completos. Con esto busco mostrarle al modelo no solo qué tiene que hacer, sino también **cómo quiero que entregue la respuesta**.

Para EmprendeCopy esto me resulta útil porque necesito que distintos negocios reciban una salida con una estructura bastante parecida.


In [9]:
prompt_few_shot = f"""
Sos especialista en copywriting para pequeños negocios.

Generá contenido promocional y respondé EXACTAMENTE con esta estructura:

Título: ...
Copy: ...
CTA: ...
Hashtags: #... #... #... #... #...

Reglas:
- El copy debe tener como máximo 80 palabras.
- Debe haber exactamente 5 hashtags.
- No inventes precios, condiciones ni características no proporcionadas.
- No agregues explicaciones antes ni después.

EJEMPLO 1
Negocio: Barbería
Servicio: Corte + barba
Promoción: 15 % de descuento los martes
Público: Hombres de 18 a 40 años
Plataforma: Instagram
Tono: Cercano
Objetivo: Reservar turnos

Título: Martes de estilo
Copy: Renová tu look con corte y barba y aprovechá un 15 % de descuento todos los martes. Una propuesta simple para mantener tu estilo con atención profesional.
CTA: Reservá tu turno para este martes.
Hashtags: #Barbería #Corte #Barba #Estilo #Promoción

EJEMPLO 2
Negocio: Cafetería
Servicio: Combo de café y medialunas
Promoción: Precio especial de 8 a 11 h
Público: Personas que trabajan o estudian en la zona
Plataforma: Instagram
Tono: Cálido
Objetivo: Aumentar visitas por la mañana

Título: Empezá la mañana con algo rico
Copy: Hacé una pausa y disfrutá nuestro combo de café y medialunas con precio especial de 8 a 11 h. Ideal para arrancar el día antes del trabajo o el estudio.
CTA: Pasá esta mañana y disfrutá el combo.
Hashtags: #Café #Desayuno #Medialunas #Cafetería #BuenDía

CASO NUEVO
Negocio: {caso['tipo_negocio']}
Servicio: {caso['producto_servicio']}
Promoción: {caso['promocion']}
Público: {caso['publico']}
Plataforma: {caso['plataforma']}
Tono: {caso['tono']}
Objetivo: {caso['objetivo']}
"""

respuesta_few, metricas_few = chat(prompt_few_shot, temperature=0)

print("=== RESPUESTA FEW-SHOT ===")
print(respuesta_few)
print("\n=== USO ===")
print(metricas_few)


=== RESPUESTA FEW-SHOT ===
Título: Agosto de color  
Copy: Descubrí tu nuevo look con nuestra coloración y aprovechá un 20 % de descuento durante todo agosto. Un servicio profesional que resalta tu estilo y cuida tu cabello.  
CTA: Reservá tu turno y luce espectacular.  
Hashtags: #Peluquería #Coloración #Descuento #Agosto #Mujeres💇‍♀️

=== USO ===
{'input_tokens': 542, 'output_tokens': 110, 'total_tokens': 652, 'costo_estimado_usd': 7.365e-05}


## 9. Comparación del formato

Para comparar las dos pruebas me enfoqué en cosas fáciles de comprobar de forma automática:

- si aparece un título;
- si aparece el copy;
- si aparece la CTA;
- si aparece la sección de hashtags;
- si devuelve exactamente cinco hashtags.

No intento medir si un texto es “más creativo” que otro, sino verificar si cada prompt respeta mejor la estructura que necesito.

Esta comprobación se hace directamente con Python, así que no requiere otra consulta a la API.


In [10]:
import re

def evaluar_respuesta(texto):
    hashtags = re.findall(r"(?<!\w)#[\wÁÉÍÓÚÜÑáéíóúüñ]+", texto)

    return {
        "tiene_titulo": "título:" in texto.lower() or "titulo:" in texto.lower(),
        "tiene_copy": "copy:" in texto.lower(),
        "tiene_cta": "cta:" in texto.lower(),
        "tiene_hashtags": "hashtags:" in texto.lower(),
        "cantidad_hashtags_detectados": len(hashtags),
        "cumple_5_hashtags": len(hashtags) == 5,
    }

evaluacion_zero = evaluar_respuesta(respuesta_zero)
evaluacion_few = evaluar_respuesta(respuesta_few)

print("ZERO-SHOT")
print(json.dumps(evaluacion_zero, indent=2, ensure_ascii=False))

print("\nFEW-SHOT")
print(json.dumps(evaluacion_few, indent=2, ensure_ascii=False))


ZERO-SHOT
{
  "tiene_titulo": false,
  "tiene_copy": false,
  "tiene_cta": false,
  "tiene_hashtags": false,
  "cantidad_hashtags_detectados": 10,
  "cumple_5_hashtags": false
}

FEW-SHOT
{
  "tiene_titulo": true,
  "tiene_copy": true,
  "tiene_cta": true,
  "tiene_hashtags": true,
  "cantidad_hashtags_detectados": 5,
  "cumple_5_hashtags": true
}


### Escala de evaluación (0 a 10)

Además de la comprobación automática anterior, agregué una escala simple para poder comparar las dos salidas con los mismos criterios.

Cada criterio vale **2 puntos**, para un total de **10**:

| Criterio | Máximo | Zero-shot | Few-shot |
|---|---:|---:|---:|
| Respeta la estructura solicitada (Título, Copy, CTA y Hashtags) | 2 | 0 | 2 |
| Devuelve exactamente 5 hashtags | 2 | 0 | 2 |
| El copy se mantiene dentro del máximo de 80 palabras | 2 | 2 | 2 |
| Incluye una CTA clara y accionable | 2 | 2 | 2 |
| No agrega canales, condiciones o datos que no fueron proporcionados | 2 | 0 | 2 |
| **Puntaje total** | **10** | **4/10** | **10/10** |

En el caso **zero-shot**, el contenido es útil, pero no respeta las etiquetas de la estructura, devuelve 10 hashtags y agrega la indicación de hacer clic en el enlace de la bio o enviar un DM, datos que yo no había proporcionado.

En **few-shot**, la salida respeta la estructura pedida, devuelve los 5 hashtags y no agrega canales de contacto que no estén en los datos de entrada.

Con esta escala no intento decidir cuál texto es “más creativo”, sino medir con reglas concretas cuánto se acerca cada salida al formato que necesito para EmprendeCopy.


### Qué resultado me deja la comparación

La idea de esta prueba es ver si **dar ejemplos concretos** ayuda a controlar mejor la respuesta.

En este caso, el zero-shot también puede obtener un buen resultado porque el modelo ya es capaz de seguir instrucciones bastante claras. Aun así, el few-shot me permite dejar mucho más marcado el formato y el estilo que quiero repetir cuando cambien los datos del negocio.

También hay una contra: al agregar ejemplos, el prompt es más largo y usa más tokens de entrada. Por eso no siempre conviene usar few-shot por defecto; depende de si esa mejora en consistencia justifica el costo extra.


# Parte B — Texto a imagen e iteración

## 10. Prueba visual de la Preentrega 1

En la primera entrega usé este prompt para generar la imagen:

> **Professional advertising photograph of a modern women's hair salon, stylish woman with freshly colored glossy hair, elegant minimalist interior, medium close-up composition, warm soft lighting, sophisticated neutral color palette, realistic textures, clean premium aesthetic, empty space for promotional copy, square Instagram composition.**

Para esta prueba visual usé la **herramienta de generación de imágenes de ChatGPT**. La generación la hice de forma manual: Groq genera o ayuda a mejorar el prompt visual y después yo uso ese prompt en la herramienta de imágenes.

Por ese motivo, la POC no realiza una llamada automatizada adicional a una API de imágenes. Las consultas que cuento y comparo en la notebook corresponden solamente a Groq.

Este fue el resultado:


![Imagen generada para la prueba visual](assets/peluqueria-generada.png)

### Qué pasó con la imagen

En general, la imagen cumple bastante bien con lo que buscaba:

- parece una pieza publicitaria;
- se entiende que se trata de una peluquería;
- el cabello es uno de los elementos principales;
- usa una iluminación cálida;
- mantiene un formato cuadrado;
- deja una zona con espacio visual.

El problema es que apareció el texto **“LUMIÈRE HAIR SALON”**.

En el prompt había pedido espacio para agregar el copy después, pero no había dejado suficientemente claro que la imagen no debía generar letras, marcas o carteles.

### Cómo lo corregiría

Para una nueva generación agregaría una instrucción más explícita:

> **Do not generate letters, words, typography, logos, signs, brand names or readable text anywhere in the image. Keep the left side completely clean and blank for copy to be added later during graphic design.**

Este caso me sirve para mostrar algo importante del prompting: no siempre el primer resultado sale exactamente como uno espera. Se puede revisar qué falló, ajustar una parte concreta del prompt y volver a probar.


# Parte C — Versión final optimizada

## 11. Copy + prompt visual en una sola consulta

Las pruebas anteriores están separadas porque necesito comparar las técnicas.

Pero si EmprendeCopy se usara realmente, no tendría sentido ejecutar primero zero-shot, después few-shot y recién después generar el resultado final.

Por eso armé una versión que directamente devuelve en **una sola consulta**:

- título;
- copy;
- CTA;
- hashtags;
- prompt visual.

Para que la salida sea fácil de usar y mantenga siempre los mismos campos, utilizo **JSON Schema / Structured Outputs**.


In [11]:
ESQUEMA_EMPRENDECOPY = {
    "type": "json_schema",
    "json_schema": {
        "name": "emprendecopy_output",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "titulo": {"type": "string"},
                "copy": {"type": "string"},
                "cta": {"type": "string"},
                "hashtags": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "image_prompt": {"type": "string"}
            },
            "required": ["titulo", "copy", "cta", "hashtags", "image_prompt"],
            "additionalProperties": False
        }
    }
}


def generar_emprendecopy(datos):
    """
    Genera el contenido de EmprendeCopy en una sola llamada
    y devuelve también los datos de uso de la API.
    """
    instrucciones = """
Sos especialista en copywriting y prompts visuales para pequeños negocios.

A partir de los datos recibidos:
1. Creá un título breve.
2. Creá un copy de máximo 80 palabras.
3. Creá una CTA concreta.
4. Devolvé exactamente 5 hashtags.
5. Creá un image_prompt en inglés, de máximo 70 palabras, para una imagen publicitaria.

El image_prompt debe definir sujeto, entorno, estilo, encuadre, iluminación,
paleta y composición. Debe dejar espacio limpio para agregar copy posteriormente.
No debe pedir letras, palabras, tipografía, logos, carteles, marcas ni texto legible.

No inventes precios, descuentos, fechas, servicios ni características no presentes
en los datos del usuario.
"""

    respuesta = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": instrucciones},
            {
                "role": "user",
                "content": json.dumps(datos, ensure_ascii=False)
            },
        ],
        reasoning_effort="low",
        max_completion_tokens=1200,
        response_format=ESQUEMA_EMPRENDECOPY,
    )

    resultado = json.loads(respuesta.choices[0].message.content)
    metricas = estimar_costo(respuesta.usage)

    # Como control extra, reviso que hayan vuelto exactamente cinco hashtags.
    if len(resultado.get("hashtags", [])) != 5:
        print(
            "Advertencia: se esperaban 5 hashtags y se recibieron",
            len(resultado.get("hashtags", []))
        )

    return resultado, metricas


## 12. Prueba de la versión final

Ahora ejecuto esa versión optimizada usando el mismo caso de la peluquería.

Esta sería la forma que usaría normalmente EmprendeCopy, porque ya no necesita hacer las comparaciones anteriores cada vez.


In [12]:
resultado_final, metricas_final = generar_emprendecopy(caso)

print("=== RESULTADO FINAL ===")
print(json.dumps(resultado_final, indent=2, ensure_ascii=False))

print("\n=== USO Y COSTO APROXIMADO ===")
print(json.dumps(metricas_final, indent=2, ensure_ascii=False))


=== RESULTADO FINAL ===
{
  "titulo": "¡Brilla con tu nuevo color!",
  "copy": "En nuestra peluquería femenina, la coloración es un arte que resalta tu estilo. Este agosto, disfruta de un 20 % de descuento y déjate transformar por expertos que combinan tendencia y cuidado. Reserva tu cita ahora y vive la experiencia que solo nuestras manos pueden crear.",
  "cta": "Reserva tu turno hoy y luce tu mejor color",
  "hashtags": [
    "#ColorConEstilo",
    "#PeluqueriaFemenina",
    "#AgostoBrillante",
    "#CorteYColor",
    "#MujeresConEstilo"
  ],
  "image_prompt": "A stylish woman in a modern salon, mid‑action hairdresser applying color, studio setting with natural light filtering through large windows, pastel palette of soft blues and pinks, close‑up shot, clean background with minimal clutter, composition balanced with subject slightly off‑center, highlighting the vibrant hair color, leaving a generous blank space for text overlay"
}

=== USO Y COSTO APROXIMADO ===
{
  "input_tokens":

## 13. Cantidad de consultas y costo

Si ejecuto toda la notebook para mostrar el experimento, hago:

- 1 consulta para zero-shot;
- 1 consulta para few-shot;
- 1 consulta para la versión final.

**En total son 3 consultas durante la demostración.**

Las dos primeras sirven para comparar las técnicas y no formarían parte del uso normal del proyecto.

### En un uso real

**Datos del negocio → 1 consulta a Groq → copy + CTA + hashtags + prompt visual**

Por eso, una vez elegido el prompt final, EmprendeCopy necesita **una sola consulta de texto por publicación**.


In [13]:
def mostrar_resumen_costos():
    filas = [
        ("Zero-shot (experimento)", metricas_zero),
        ("Few-shot (experimento)", metricas_few),
        ("Versión final (uso real)", metricas_final),
    ]

    print(f"{'Etapa':30} {'Tokens':>10} {'Costo USD aprox.':>18}")
    print("-" * 62)

    for nombre, m in filas:
        print(
            f"{nombre:30} "
            f"{m['total_tokens']:>10} "
            f"{m['costo_estimado_usd']:>18.8f}"
        )

mostrar_resumen_costos()


Etapa                              Tokens   Costo USD aprox.
--------------------------------------------------------------
Zero-shot (experimento)               340         0.00006420
Few-shot (experimento)                652         0.00007365
Versión final (uso real)              613         0.00009480


# Parte D — Prueba con otros datos

## 14. Opción interactiva

Como agregado, dejé una función para ingresar los datos de otro negocio directamente desde la notebook.

La función reutiliza la misma versión final, así que no cambia la lógica del proyecto.

Las líneas que hacen la consulta están comentadas para que esta parte no genere llamadas extra por accidente. Si se descomentan y se ejecutan, cada prueba nueva hace **una consulta a Groq**.


In [14]:
def pedir_datos():
    print("Ingresá los datos del negocio:\n")

    return {
        "tipo_negocio": input("Tipo de negocio: ").strip(),
        "producto_servicio": input("Producto o servicio: ").strip(),
        "promocion": input("Promoción: ").strip(),
        "publico": input("Público objetivo: ").strip(),
        "plataforma": input("Plataforma: ").strip(),
        "tono": input("Tono: ").strip(),
        "objetivo": input("Objetivo: ").strip(),
    }


# Si quiero probar otro negocio, descomento estas tres líneas:
# mis_datos = pedir_datos()
# mi_resultado, mi_uso = generar_emprendecopy(mis_datos)
# print(json.dumps(mi_resultado, indent=2, ensure_ascii=False))


# 15. Conclusiones

Después de hacer las pruebas, considero que EmprendeCopy se puede resolver de forma bastante simple con las herramientas vistas en el curso.

### Lo principal que me dejó esta POC

- Con **zero-shot** puedo obtener una primera respuesta sin darle ejemplos al modelo.
- Con **few-shot** puedo mostrarle ejemplos del formato que quiero y hacer más consistente la salida. En la escala que definí, el resultado pasó de **4/10 en zero-shot a 10/10 en few-shot**.
- La prueba de imagen mostró que también es importante revisar el resultado y **ajustar el prompt** cuando aparece algo que no pedí.
- Para demostrar las técnicas necesité varias llamadas, pero no hace falta repetirlas en el funcionamiento normal.
- Con **Structured Outputs** puedo recibir los datos en una estructura fija y más fácil de reutilizar.
- La versión final puede generar el contenido textual y el prompt para la imagen con **una sola consulta a Groq por publicación**.

En general, las nuevas técnicas me permitieron mejorar la propuesta de la primera entrega, sobre todo en el control del formato y en la reducción de consultas innecesarias.


---

## Referencias técnicas

- Groq — GPT OSS 20B: https://console.groq.com/docs/model/openai/gpt-oss-20b
- Groq — Structured Outputs: https://console.groq.com/docs/structured-outputs
- Groq — Modelos disponibles: https://console.groq.com/docs/models

**Nota:** para la estimación de costos usé las tarifas consultadas el 23/08/2026. Como pueden cambiar, habría que volver a verificarlas si el proyecto se llevara a un uso real.
